<a href="https://colab.research.google.com/github/hyperpipe-kr/colab-examples/blob/main/06_bart_summarization_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 문서 요약 모델 — facebook/bart-large-cnn (Colab)

BART-large를 CNN/DailyMail 데이터로 파인튜닝한 **추상적 요약(abstractive summarization)** 모델입니다.

| 항목 | 값 |
|---|---|
| 파라미터 | 약 0.4B |
| 입력 한계 | **1024 토큰** (초과분은 잘림) |
| 언어 | **영어 전용** |
| 라이선스 | MIT |

> ⚠️ 한국어 문서를 요약하려면 이 모델은 적합하지 않습니다. 맨 아래 "한국어 문서" 섹션을 참고하세요.

**런타임 → 런타임 유형 변경 → T4 GPU** 로 설정하고 시작하세요. (CPU로도 돌아가지만 문단당 10~30초 걸립니다.)


## 1. 설치 및 환경 확인

In [1]:
!pip -q install transformers==4.44.0 torch accelerate sentencepiece

import torch, transformers
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU 사용)")

transformers: 4.44.0
torch: 2.11.0+cu128
GPU: Tesla T4


## 2. 가장 빠른 방법 — pipeline

모델 가중치 약 1.6GB를 내려받으므로 첫 실행은 1~2분 걸립니다.

In [ ]:
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=DEVICE,
    torch_dtype=torch.float16 if DEVICE == 0 else torch.float32,
)

ARTICLE = """
The James Webb Space Telescope has captured the deepest and sharpest infrared image
of the distant universe to date. Known as Webb's First Deep Field, this image of
galaxy cluster SMACS 0723 is overflowing with detail. Thousands of galaxies,
including the faintest objects ever observed in the infrared, have appeared in
Webb's view for the first time. This slice of the vast universe covers a patch of
sky approximately the size of a grain of sand held at arm's length by someone on
the ground. The combined mass of this galaxy cluster acts as a gravitational lens,
magnifying much more distant galaxies behind it. Researchers said the image was
assembled from composites made at different wavelengths, achieved in 12.5 hours,
far less time than the weeks required for comparable Hubble Deep Field images.
"""

result = summarizer(ARTICLE, max_length=130, min_length=30, do_sample=False, truncation=True)

print('##########')
print(result[0])
print('##########')
print(result[0]["summary_text"])



/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


##########
{'summary_text': "Thousands of galaxies, including the faintest objects ever observed in the infrared, have appeared in Webb's view for the first time. Researchers said the image was assembled from composites made at different wavelengths, achieved in 12.5 hours."}
##########
Thousands of galaxies, including the faintest objects ever observed in the infrared, have appeared in Webb's view for the first time. Researchers said the image was assembled from composites made at different wavelengths, achieved in 12.5 hours.


## 3. 생성 파라미터 정리

요약 품질은 대부분 여기서 갈립니다.

| 파라미터 | 설명 | 권장값 |
|---|---|---|
| `max_length` / `min_length` | 요약문의 **토큰** 길이 상한/하한 | 원문의 20~30% 수준 |
| `num_beams` | 빔 서치 폭. 클수록 품질↑ 속도↓ | 4 |
| `length_penalty` | >1이면 긴 요약, <1이면 짧은 요약 선호 | 1.0~2.0 |
| `no_repeat_ngram_size` | 같은 n-gram 반복 금지 | 3 |
| `do_sample` | False = 결정론적(요약에 권장) | False |
| `truncation` | 1024 토큰 초과 시 자동 절단 | True |


In [ ]:
GEN_KWARGS = dict(
    max_length=142,
    min_length=40,
    num_beams=4,
    length_penalty=2.0,
    no_repeat_ngram_size=3,
    do_sample=False,
    truncation=True,
)

print(summarizer(ARTICLE, **GEN_KWARGS)[0]["summary_text"])


Thousands of galaxies, including the faintest objects ever observed in the infrared, have appeared in Webb's view for the first time. Researchers said the image was assembled from composites made at different wavelengths, achieved in 12.5 hours.


## 4. 1024 토큰이 넘는 긴 문서 처리

BART 인코더는 최대 1024 토큰까지만 봅니다. 그냥 넣으면 **뒷부분은 통째로 버려집니다.**
긴 문서는 잘라서 각각 요약한 뒤(map), 그 요약들을 다시 요약(reduce)하는 방식이 표준입니다.

문장 경계에서 자르기 위해 문장 단위로 모아 청크를 만듭니다.

In [ ]:
import re
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")

def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    return re.split(r"(?<=[.!?])\s+", text)

def chunk_by_tokens(text, max_tokens=900):
    """문장 경계를 지키면서 max_tokens 이하 청크로 나눈다."""
    chunks, cur, cur_len = [], [], 0
    for sent in split_sentences(text):
        n = len(tokenizer.encode(sent, add_special_tokens=False))
        if cur and cur_len + n > max_tokens:
            chunks.append(" ".join(cur))
            cur, cur_len = [sent], n
        else:
            cur.append(sent)
            cur_len += n
    if cur:
        chunks.append(" ".join(cur))
    return chunks

def summarize_long(text, chunk_tokens=900, batch_size=4, reduce=True):
    chunks = chunk_by_tokens(text, chunk_tokens)
    print(f"청크 {len(chunks)}개로 분할")

    partials = summarizer(chunks, batch_size=batch_size, **GEN_KWARGS)
    partials = [p["summary_text"].strip() for p in partials]

    if not reduce or len(partials) == 1:
        return " ".join(partials)

    merged = " ".join(partials)
    # 합친 결과가 또 1024를 넘으면 재귀적으로 한 번 더 줄인다
    if len(tokenizer.encode(merged, add_special_tokens=False)) > chunk_tokens:
        return summarize_long(merged, chunk_tokens, batch_size, reduce=True)
    return summarizer(merged, **GEN_KWARGS)[0]["summary_text"].strip()


long_text = ARTICLE * 12   # 데모용: 실제로는 여기에 긴 원문을 넣으세요
print(summarize_long(long_text))


청크 3개로 분할
Thousands of galaxies, including the faintest objects ever observed in the infrared, have appeared in Webb's view for the first time. Researchers said the image was assembled from composites made at different wavelengths, achieved in 12.5 hours.


## 5. 파일 업로드해서 요약하기

`.txt` 또는 `.pdf`를 올려 바로 요약합니다.

In [ ]:
!pip -q install pypdf

from google.colab import files
from pypdf import PdfReader
import io, os

uploaded = files.upload()   # 파일 선택 창이 뜹니다

for name, data in uploaded.items():
    ext = os.path.splitext(name)[1].lower()
    if ext == ".pdf":
        reader = PdfReader(io.BytesIO(data))
        text = "\n".join((page.extract_text() or "") for page in reader.pages)
    else:
        text = data.decode("utf-8", errors="ignore")

    print(f"\n===== {name} ({len(text):,}자) =====")
    print(summarize_long(text))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 12.2 MB/s eta 0:00:00


Saving 모델_서빙_개요.pdf to 모델_서빙_개요.pdf

===== 모델_서빙_개요.pdf (2,127자) =====
청크 4개로 분할


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 6. 여러 문서 일괄 요약 (배치)

리스트를 그대로 넘기면 GPU 배치로 처리됩니다. T4 기준 `batch_size=4~8`이 적당하고,
OOM이 나면 값을 줄이세요.

In [ ]:
import pandas as pd

docs = [
    "First document text ...",
    "Second document text ...",
    "Third document text ...",
]

outputs = summarizer(docs, batch_size=4, **GEN_KWARGS)

df = pd.DataFrame({
    "원문_길이": [len(d) for d in docs],
    "요약": [o["summary_text"] for o in outputs],
})
df.to_csv("summaries.csv", index=False, encoding="utf-8-sig")
df


## 7. pipeline 없이 직접 제어하기

전처리/후처리를 세밀하게 손봐야 할 때는 모델과 토크나이저를 직접 다룹니다.

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(
    "cuda" if torch.cuda.is_available() else "cpu"
).eval()

def summarize_raw(texts, **kw):
    if isinstance(texts, str):
        texts = [texts]
    inputs = tokenizer(texts, max_length=1024, truncation=True,
                       padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_length=kw.get("max_length", 142),
            min_length=kw.get("min_length", 40),
            num_beams=4,
            length_penalty=2.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tokenizer.batch_decode(ids, skip_special_tokens=True)

print(summarize_raw(ARTICLE)[0])


## 8. 요약 품질 평가 (ROUGE)

정답 요약이 있다면 ROUGE로 수치화합니다. 참고로 원 논문의 CNN/DailyMail 자체 보고 점수는
ROUGE-1 42.9 / ROUGE-2 20.8 / ROUGE-L 30.6 수준입니다.

In [ ]:
!pip -q install evaluate rouge_score

import evaluate
rouge = evaluate.load("rouge")

predictions = ["The James Webb telescope captured the deepest infrared image of the universe."]
references  = ["Webb's First Deep Field shows galaxy cluster SMACS 0723 in unprecedented detail."]

print(rouge.compute(predictions=predictions, references=references))


## 9. 한국어 문서를 요약해야 한다면

`bart-large-cnn`은 영어 코퍼스로만 학습되어 한국어 입력은 사실상 사용할 수 없습니다.
토크나이저부터 한국어를 제대로 쪼개지 못합니다. 선택지는 셋입니다.

1. **한국어 전용 요약 모델로 교체** — 아래 셀. 모델명만 바꾸면 위 코드가 그대로 동작합니다.
2. **번역 → 요약 → 번역** — 품질 손실이 누적되어 실무에는 권하지 않습니다.
3. **LLM API 사용** — 문서가 길고 도메인 특화 요약이 필요하면 이쪽이 현실적으로 더 낫습니다.

| 모델 | 특징 |
|---|---|
| `gogamza/kobart-summarization` | KoBART 기반, 가볍고 무난한 기본값 |
| `eenzeenee/t5-base-korean-summarization` | T5 계열, 문어체 문서에 강함 |
| `lcw99/t5-base-korean-text-summary` | 뉴스/기사체 요약 |


In [3]:
from transformers import pipeline

DEVICE = 0 if torch.cuda.is_available() else -1


ko_summarizer = pipeline(
    "summarization",
    model="gogamza/kobart-summarization",
    device=DEVICE,
)

KO_TEXT = """
제임스 웹 우주망원경이 지금까지 촬영된 것 중 가장 깊고 선명한 적외선 우주 이미지를 공개했다.
은하단 SMACS 0723을 담은 이 사진에는 적외선으로 관측된 가장 희미한 천체를 포함해 수천 개의
은하가 담겼다. 연구진은 서로 다른 파장에서 촬영한 합성 이미지를 12.5시간 만에 완성했으며,
이는 유사한 허블 딥 필드 이미지에 수 주가 걸렸던 것과 비교하면 크게 단축된 시간이라고 밝혔다.
"""

print(ko_summarizer(KO_TEXT, max_length=128, min_length=32, do_sample=False)[0]["summary_text"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.
You passed along `num_labe

 제임스 웹 우주망원경이 지금까지 촬영된 것 중 가장 깊고 선명한 적외선 우주 이미지를 카메라 촬영된 것 중 가장 깊고 선명한 적외선 우주 이미지를 카메라 촬영된 것 중 가장 깊고 선명한 적외선 우주 이미지를 카메라 촬영된 것 중 가장 희미한 천체를 포함해 수천 개의개의은하가 담겼다.


## 10. 자주 부딪히는 문제

- **요약이 원문 앞부분만 반영됨** → 1024 토큰 초과. 4번의 `summarize_long` 사용.
- **CUDA out of memory** → `batch_size` 축소, `torch_dtype=torch.float16`, 런타임 재시작.
- **요약이 너무 짧음/김** → `min_length`/`max_length`와 `length_penalty` 조정.
- **같은 문장 반복** → `no_repeat_ngram_size=3` 확인, `num_beams` 상향.
- **원문에 없는 내용이 생성됨** → 추상적 요약 모델의 구조적 한계(hallucination). 사실 정확도가
  중요하면 추출적 요약이나 LLM + 근거 인용 방식을 함께 검토하세요.
